**Spark SQL**

**Temp Views — Bridging DataFrames and SQL**

A temp view registers a DataFrame under a name so you can query it using SQL with spark.sql(). This is incredibly useful — you can write SQL for the parts that are clearer in SQL, and DataFrame code for the parts that need programmatic logic.

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-20")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1be95625-c4f8-47f6-b2ee-2fbfdd049fa1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 133ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
# Register a DataFrame as a temp view
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

# Now query it with plain SQL
result = spark.sql("""
    SELECT order_id, customer_id, unit_price, status
    FROM orders
    WHERE unit_price > 500
    ORDER BY unit_price DESC
""")

result.show(5)

+--------+-----------+----------+---------+
|order_id|customer_id|unit_price|   status|
+--------+-----------+----------+---------+
|   O0034|       C014|   1299.99|Delivered|
|   O0061|       C016|   1299.99|  Shipped|
|   O0009|       C009|   1299.99|Delivered|
|   O0041|       C021|   1299.99|Delivered|
|   O0024|       C004|   1299.99|Delivered|
+--------+-----------+----------+---------+
only showing top 5 rows


**createOrReplaceTempView() vs createTempView()**

createOrReplaceTempView() overwrites the view if it already exists — no error. createTempView() throws an error if the view name is already taken. Always use createOrReplaceTempView() in notebooks to avoid errors when re-running cells.

In [3]:
# Join + aggregate in SQL — identical logic to PySpark groupBy/join
result = spark.sql("""
    SELECT
        c.segment,
        COUNT(o.order_id) AS order_count,
        ROUND(SUM(o.unit_price), 2) AS total_revenue,
        ROUND(AVG(o.unit_price), 2) AS avg_order_value
    FROM orders o
    INNER JOIN customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.segment
    ORDER BY total_revenue DESC
""")

result.show()

+----------+-----------+-------------+---------------+
|   segment|order_count|total_revenue|avg_order_value|
+----------+-----------+-------------+---------------+
|Enterprise|         37|     11549.63|         312.15|
|       SMB|         36|     11269.64|         313.05|
|   Startup|         27|      8974.73|          332.4|
+----------+-----------+-------------+---------------+



In [4]:
# Top order per region using SQL window function
result = spark.sql("""
    SELECT region, order_id, unit_price, rnk
    FROM (
        SELECT
            region, order_id, unit_price,
            RANK() OVER (PARTITION BY region ORDER BY unit_price DESC) AS rnk
        FROM orders
    )
    WHERE rnk = 1
""")

result.show()

+-------+--------+----------+---+
| region|order_id|unit_price|rnk|
+-------+--------+----------+---+
|   East|   O0001|   1299.99|  1|
|   East|   O0051|   1299.99|  1|
|Midwest|   O0034|   1299.99|  1|
|Midwest|   O0061|   1299.99|  1|
|Midwest|   O0094|   1299.99|  1|
|  South|   O0009|   1299.99|  1|
|  South|   O0024|   1299.99|  1|
|   West|   O0041|   1299.99|  1|
|   West|   O0072|   1299.99|  1|
|   West|   O0080|   1299.99|  1|
|   West|   O0088|   1299.99|  1|
+-------+--------+----------+---+



## SQL vs DataFrame API — When to Use Which

| Use SQL When | Use DataFrame API When |
|---------------|------------------------|
| The query is naturally expressed as SQL (joins, aggregations, complex filters). | You need programmatic logic (loops, conditionals, dynamic column generation, reusable functions). |
| Your team is SQL-first and more comfortable writing SQL queries. | You're building ETL/data pipelines with Python control flow. |
| Migrating existing SQL queries to Spark with minimal changes. | You need dynamic queries based on runtime conditions or user input. |
| Writing ad-hoc analysis or exploratory queries. | You want better IDE support, autocompletion, and type-safe column references. |
| Business analysts or SQL developers will maintain the code. | You need to create reusable functions or modular transformations. |
| The logic is mostly declarative and easy to express in SQL. | You're integrating Spark with other Python libraries or application logic. |

## Managed vs External Tables

Beyond temporary views, Spark supports **persistent tables** backed by a **metastore**. These tables are of two types: **Managed Tables** and **External Tables**.

| Managed Table | External Table |
|----------------|----------------|
| Spark manages both the **metadata** and the **data files**. | Spark manages only the **metadata**. |
| Dropping the table deletes both the metadata and the underlying data. | Dropping the table removes only the metadata; the data files remain untouched. |
| Data is stored in Spark's default warehouse directory (for example, `spark-warehouse`). | Data is stored at a user-specified location, such as an S3 bucket or HDFS path. |
| Suitable for development, testing, or temporary datasets. | Preferred for production environments where data must outlive the table definition. |
| Data lifecycle is controlled by Spark. | Data lifecycle is controlled by you or your organization. |
| Created without specifying a storage location. | Created by specifying a storage location using the `LOCATION` clause. |

### Quick Comparison

| Feature | Managed Table | External Table |
|---------|---------------|----------------|
| Metadata | Managed by Spark | Managed by Spark |
| Data Files | Managed by Spark | Managed by User |
| `DROP TABLE` | Deletes metadata + data | Deletes metadata only |
| Storage Location | Spark warehouse | User-defined location (S3, HDFS, etc.) |
| Production Usage | Less common | Most common |

 **AWS Best Practice:** In AWS environments (EMR, Glue, Athena),

 **External Tables** are the standard choice because the data is typically stored in 
 
 **Amazon S3**. This allows multiple services (Spark, Athena, Glue, Redshift Spectrum, etc.) to access the same data without moving or duplicating it.

In [5]:
# External table — points to data already in S3
spark.sql("""
CREATE TABLE IF NOT EXISTS orders_external
USING PARQUET
LOCATION 's3a://pyspark-30-days-rahul-2026/data/orders_output/'
""")

spark.sql("SELECT * FROM orders_external LIMIT 5").show()

# Dropping this table removes only the metadata — your S3 data is untouched
spark.sql("DROP TABLE orders_external")

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|        10.0|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|         0.0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|        15.0|Delivered|   Credit Card|Midwest|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|         5.0|Delivered|    Debit Card|  South|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|         0.0|Delivered|   Credit Card|   West|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+



DataFrame[]

**Task 1**
Register orders.csv and products.csv as temp views. Write a SQL query that joins them and shows total revenue per category, sorted descending.

In [6]:
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
spark.sql("""select
p.category,round(sum(o.unit_price*o.quantity),2) as total_revenue
from orders o
join products p
on o.product_id=p.product_id
group by p.category
order by total_revenue desc""").show()


+---------------+-------------+
|       category|total_revenue|
+---------------+-------------+
|    Electronics|     35088.37|
|      Furniture|      9009.39|
|Office Supplies|       284.92|
+---------------+-------------+



**Task 2**

Using SQL, write a query that finds the top 3 most expensive orders per region using RANK() OVER (PARTITION BY ... ORDER BY ...).

In [14]:
result=spark.sql("""with cte as (
    select region,product_id,rank() over (partition by region order by unit_price desc) as rank
    from orders
)
select region,product_id,rank from cte where rank <= 3
""").show()

+-------+----------+----+
| region|product_id|rank|
+-------+----------+----+
|   East|      P001|   1|
|   East|      P001|   1|
|   East|      P020|   3|
|Midwest|      P001|   1|
|Midwest|      P001|   1|
|Midwest|      P001|   1|
|  South|      P001|   1|
|  South|      P001|   1|
|  South|      P020|   3|
|   West|      P001|   1|
|   West|      P001|   1|
|   West|      P001|   1|
|   West|      P001|   1|
+-------+----------+----+



**Task 3**

Write the same query from Task 2 using the PySpark DataFrame API instead of SQL. Compare both approaches — which do you find more readable?

In [16]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
orders_df.withColumn('rk',F.rank().over(Window.partitionBy('region').orderBy(F.desc('unit_price')))).\
    filter(F.col('rk')<=3).\
    select("region", "product_id", "rk").show()

+-------+----------+---+
| region|product_id| rk|
+-------+----------+---+
|   East|      P001|  1|
|   East|      P001|  1|
|   East|      P020|  3|
|Midwest|      P001|  1|
|Midwest|      P001|  1|
|Midwest|      P001|  1|
|  South|      P001|  1|
|  South|      P001|  1|
|  South|      P020|  3|
|   West|      P001|  1|
|   West|      P001|  1|
|   West|      P001|  1|
|   West|      P001|  1|
+-------+----------+---+



**Task 4**

Create an external table pointing to a Parquet file you wrote earlier in S3. Query it with SQL. Then drop the table and verify your data in S3 is still intact.

In [23]:
spark.sql("""
CREATE TABLE IF NOT EXISTS orders_external
USING PARQUET
LOCATION 's3a://pyspark-30-days-rahul-2026/data/orders_output/'


""")
spark.sql("""
SELECT customer_id,
       COUNT(*) AS total_orders
FROM orders_external
GROUP BY customer_id
ORDER BY total_orders DESC
""").show()

spark.sql("DROP TABLE IF EXISTS orders_external")

spark.read.parquet(
    "s3a://pyspark-30-days-rahul-2026/data/orders_output/"
).show(5, truncate=False)

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C003|           5|
|       C004|           5|
|       C005|           5|
|       C001|           5|
|       C002|           5|
|       C006|           4|
|       C010|           4|
|       C007|           4|
|       C018|           4|
|       C012|           4|
|       C015|           4|
|       C020|           4|
|       C019|           4|
|       C011|           4|
|       C014|           4|
|       C017|           4|
|       C009|           4|
|       C008|           4|
|       C013|           4|
|       C016|           4|
+-----------+------------+
only showing top 20 rows


+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|O0001   |C001       |P001      |2023-01-05|2       |1299.99   |10.0        |Delivered|Credit Card   |East   |
|O0002   |C002       |P005      |2023-01-07|1       |449.99    |0.0         |Delivered|PayPal        |West   |
|O0003   |C003       |P003      |2023-01-10|4       |349.99    |15.0        |Delivered|Credit Card   |Midwest|
|O0004   |C004       |P006      |2023-01-12|2       |89.99     |5.0         |Delivered|Debit Card    |South  |
|O0005   |C005       |P002      |2023-01-15|3       |29.99     |0.0         |Delivered|Credit Card   |West   |
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
o